# Telemetry Data Analysis

This notebook analyzes player telemetry data to calculate play duration and identify players with short sessions.

**Assumption**: Each telemetry row represents a 30-second window.

In [40]:
%pip install pandas
import pandas as pd
import os

Note: you may need to restart the kernel to use updated packages.


In [41]:
# Configuration
TELEMETRY_PATH = 'data/telemetry_phase_2.telemetries.csv'
USERS_PATH = 'data/telemetry_phase_2.users.csv'
OUTPUT_PATH = 'data/processed/processed_telemetry_data.csv'

X_MINUTES = 30 # Minimum Threshold: Filter players <= X
Y_MINUTES = 45 # Maximum Cap: Filter players > Y (Trim unless rich)

# Columns to check for "Richness"
RICH_COLUMNS = ['itemsCollected', 'kills', 'rawJson.deaths']

In [42]:
# Load Data
if not os.path.exists(TELEMETRY_PATH) or not os.path.exists(USERS_PATH):
    print(f"Error: Data files not found.")
    print(f"Looking for: {TELEMETRY_PATH}")
    print(f"Looking for: {USERS_PATH}")
else:
    df_telemetry = pd.read_csv(TELEMETRY_PATH)
    df_users = pd.read_csv(USERS_PATH)
    print(f"Successfully loaded {len(df_telemetry)} telemetry rows and {len(df_users)} users.")

Successfully loaded 3348 telemetry rows and 59 users.


In [43]:
# Process Data
if 'userId' in df_telemetry.columns and '_id' in df_users.columns:
    # 1. Calculate Duration per User (Count * 30s)
    user_activity = df_telemetry.groupby('userId').size().reset_index(name='data_points')
    user_activity['duration_seconds'] = user_activity['data_points'] * 30
    user_activity['duration_minutes'] = user_activity['duration_seconds'] / 60

    # 2. Merge with User Details
    merged_data = pd.merge(
        user_activity,
        df_users[['_id', 'firstName', 'lastName']],
        left_on='userId',
        right_on='_id',
        how='inner'
    )
    
    print("Data processed and merged successfully.")
    display(merged_data.head())
else:
    print("Error: Missing required columns ('userId' in telemetry, '_id' in users).")

Data processed and merged successfully.


,userId,data_points,duration_seconds,duration_minutes,_id,firstName,lastName
0,6974892348d53c4152cf1421,105,3150,52.5,6974892348d53c4152cf1421,Sithmi,Kashmira
1,6974abcf315a39e91bc1e471,9,270,4.5,6974abcf315a39e91bc1e471,Nethu,Samarathunge
2,6974acd8315a39e91bc1e52f,157,4710,78.5,6974acd8315a39e91bc1e52f,Sakuna,Yasasvi
3,6974f3cc5d8e98b89f0a5d07,103,3090,51.5,6974f3cc5d8e98b89f0a5d07,Vidusara,Wijerathnayake
4,6974f5555d8e98b89f0a5fa5,58,1740,29.0,6974f5555d8e98b89f0a5fa5,Kavinda,Ravihansa


In [44]:
# Filter and Display Results (Analysis View)
if 'duration_minutes' in merged_data.columns:
    short_playtime = merged_data[merged_data['duration_minutes'] <= X_MINUTES].copy()
    target_playtime = merged_data[(merged_data['duration_minutes'] > X_MINUTES) & (merged_data['duration_minutes'] <= Y_MINUTES)].copy()
    capped_playtime = merged_data[merged_data['duration_minutes'] > Y_MINUTES].copy()
    
    # Sort all lists
    short_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)
    target_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)
    capped_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)
    merged_data.sort_values(by='duration_minutes', ascending=False, inplace=True)

    print(f"\n--- Analysis Report ---")
    print(f"Short (<= {X_MINUTES}m): {len(short_playtime)}")
    print(f"Target (> {X_MINUTES}m & <= {Y_MINUTES}m): {len(target_playtime)}")
    print(f"Long (> {Y_MINUTES}m): {len(capped_playtime)}")
    
    print(f"\n--- [Target Playtime] ({X_MINUTES} < d <= {Y_MINUTES} mins) ---")
    display(target_playtime[['firstName', 'lastName', 'duration_minutes', 'data_points']])

    print(f"\n--- [Long Playtime] (> {Y_MINUTES} mins) ---")
    display(capped_playtime[['firstName', 'lastName', 'duration_minutes', 'data_points']])

    print(f"\n--- [ALL USERS] Sorted by Duration ---")
    display(merged_data[['firstName', 'lastName', 'duration_minutes', 'data_points']])
else:
    print("Skipping analysis due to missing data.")


--- Analysis Report ---
Short (<= 30m): 26
Target (> 30m & <= 45m): 22
Long (> 45m): 8

--- [Target Playtime] (30 < d <= 45 mins) ---


,firstName,lastName,duration_minutes,data_points
24,Nisandu,Senanayake,42.0,84
15,Supun,Gamage,41.0,82
28,Dulina,Dias,41.0,82
9,Kaveesh,Jayawardana,40.5,81
41,Tharusha,Dulmini,39.0,78
17,Ranura,Indudunu,38.5,77
38,Harshana,Senadeera,38.5,77
27,Udeshika,Jayakody,37.5,75
49,Uthpala,Sarathchandra,36.5,73
22,Thimanya,Senanayake,36.0,72



--- [Long Playtime] (> 45 mins) ---


,firstName,lastName,duration_minutes,data_points
33,Ashfaaq,Ahamed,136.0,272
2,Sakuna,Yasasvi,78.5,157
11,Ravindu,Dushyantha,77.0,154
12,Nimeth,Dasanayake,66.0,132
6,Pasindu,Godakanda,57.0,114
0,Sithmi,Kashmira,52.5,105
3,Vidusara,Wijerathnayake,51.5,103
18,Shehan,Sanjula,49.0,98



--- [ALL USERS] Sorted by Duration ---


,firstName,lastName,duration_minutes,data_points
33,Ashfaaq,Ahamed,136.0,272
2,Sakuna,Yasasvi,78.5,157
11,Ravindu,Dushyantha,77.0,154
12,Nimeth,Dasanayake,66.0,132
6,Pasindu,Godakanda,57.0,114
0,Sithmi,Kashmira,52.5,105
3,Vidusara,Wijerathnayake,51.5,103
18,Shehan,Sanjula,49.0,98
24,Nisandu,Senanayake,42.0,84
28,Dulina,Dias,41.0,82


In [45]:
# Data Processing & Export Logic - IMPROVED
# 1. Filter Users: Keep ONLY users >= X minutes
valid_users = merged_data[merged_data['duration_minutes'] >= X_MINUTES]['userId'].unique()
final_rows = []

print(f"\n--- Processing {len(valid_users)} eligible users (>= {X_MINUTES} mins) ---")

Y_POINTS = int((Y_MINUTES * 60) / 30) # Max points (e.g., 45 mins * 2 = 90 points)

# Track Stats
users_below_x = len(merged_data[merged_data['duration_minutes'] < X_MINUTES])
users_between_x_y = 0
users_above_y = 0

# Helper: Calculate Richness Score
def calculate_richness(rows):
    score = 0
    for col in RICH_COLUMNS:
        if col in rows.columns:
            score += rows[col].sum()
    return score

for uid in valid_users:
    # Get user's telemetry rows, sorted by timestamp
    user_rows = df_telemetry[df_telemetry['userId'] == uid].copy()
    if 'timestamp' in user_rows.columns:
        user_rows.sort_values(by='timestamp', inplace=True)

    total_count = len(user_rows)
    
    if total_count <= Y_POINTS:
        # Case A: Within Range (X <= count <= Y). Keep ALL.
        final_rows.append(user_rows)
        users_between_x_y += 1
        # print(f"  [Keep] {uid}: {total_count} pts")
    else:
        # Case B: Exceeds Y. Compare Head vs Tail, keep Winner.
        users_above_y += 1
        
        # Head = First Y Points
        head_rows = user_rows.iloc[:Y_POINTS]
        # Tail = Last Y Points
        tail_rows = user_rows.iloc[-Y_POINTS:]
        
        head_score = calculate_richness(head_rows)
        tail_score = calculate_richness(tail_rows)
        
        if tail_score > head_score:
            final_rows.append(tail_rows)
            print(f"  [Tail Selected] User {uid}: {total_count} pts. Tail Richness ({tail_score}) > Head ({head_score})")
        else:
            final_rows.append(head_rows)
            print(f"  [Head Selected] User {uid}: {total_count} pts. Head Richness ({head_score}) >= Tail ({tail_score})")

# 3. Concatenate and Save
if final_rows:
    df_final = pd.concat(final_rows)
    df_final.to_csv(OUTPUT_PATH, index=False)
    print(f"\nProcessing Complete. Saved to: {OUTPUT_PATH}")
    print(f"Total Data Points Exported: {len(df_final)}")
    
    # 4. Summary Statistics
    print(f"\n--- Final Summary ---")
    print(f"Users Below X ({X_MINUTES}m): {users_below_x}")
    print(f"Users Between X and Y ({X_MINUTES}-{Y_MINUTES}m): {users_between_x_y}")
    print(f"Users More Than Y ({Y_MINUTES}m): {users_above_y}")
    print(f"Total Data Rows Processed: {len(df_final)}")
else:
    print("No data to export.")


--- Processing 30 eligible users (>= 30 mins) ---
  [Head Selected] User 6975de91547a2b541d1316de: 272 pts. Head Richness (11) >= Tail (0)
  [Head Selected] User 6974acd8315a39e91bc1e52f: 157 pts. Head Richness (443) >= Tail (436)
  [Tail Selected] User 69757efd2890de71026a214a: 154 pts. Tail Richness (213) > Head (185)
  [Tail Selected] User 69758a65547a2b541d10f4ed: 132 pts. Tail Richness (399) > Head (337)
  [Tail Selected] User 697502455d8e98b89f0a9f38: 114 pts. Tail Richness (203) > Head (186)
  [Head Selected] User 6974892348d53c4152cf1421: 105 pts. Head Richness (219) >= Tail (210)
  [Tail Selected] User 6974f3cc5d8e98b89f0a5d07: 103 pts. Tail Richness (349) > Head (321)
  [Head Selected] User 69759b30547a2b541d1151be: 98 pts. Head Richness (50) >= Tail (44)

Processing Complete. Saved to: data/processed/processed_telemetry_data.csv
Total Data Points Exported: 2301

--- Final Summary ---
Users Below X (30m): 26
Users Between X and Y (30-45m): 22
Users More Than Y (45m): 8
Total

# Death Event Integration

Integrate death events into telemetry data. Matches deaths to 30s telemetry windows and identifies high-death intervals.

In [46]:
# Load and Process Death Events (Refined Logic)
import pandas as pd
import os

# Paths
TELE_PATH = 'data/telemetry_phase_2.telemetries.csv'
DEATH_PATH = 'data/telemetry_phase_2.deathevents.csv'
OUT_Processed = 'data/processed/telemetry_with_deaths.csv'

print("Loading datasets for death analysis...")
if os.path.exists(TELE_PATH) and os.path.exists(DEATH_PATH):
    df_t = pd.read_csv(TELE_PATH)
    df_d = pd.read_csv(DEATH_PATH)
    
    # Standardize Timestamps
    t_col = 'timestamp' if 'timestamp' in df_t.columns else 'time'
    d_col = 'timestamp' if 'timestamp' in df_d.columns else 'time'
    
    for df, col in [(df_t, t_col), (df_d, d_col)]:
        df['num_time'] = pd.to_numeric(df[col], errors='coerce')
        if df['num_time'].isna().all():
            df['dt_time'] = pd.to_datetime(df[col], errors='coerce')
            df['num_time'] = df['dt_time'].astype('int64') // 10**9
            
    # Prepare for merge_asof (Sort required)
    df_t = df_t.sort_values('num_time')
    df_d = df_d.sort_values('num_time')
    
    # Detect Unit & Tolerance
    if len(df_t) > 0:
        sample_t = df_t['num_time'].iloc[0]
        is_ms = sample_t > 1000000000000
        tolerance = 30000 if is_ms else 30
        print(f"Time Unit: {'ms' if is_ms else 's'}, Tolerance: {tolerance}")

        # Prepare IDs
        if '_id' in df_t.columns:
            df_t['telem_unique_id'] = df_t['_id']
        else:
            df_t['telem_unique_id'] = df_t.index
            
        # MERGE ASOF
        # Map each death to the NEXT telemetry point (direction='forward')
        # This ensures 1-to-1 mapping.
        # mapped: Deaths augmented with closest Telemetry info
        mapped = pd.merge_asof(
            df_d, 
            df_t, 
            left_on='num_time', 
            right_on='num_time', 
            by='userId', 
            direction='forward', 
            tolerance=tolerance,
            suffixes=('_d', '_t')
        )
        
        # Filter valid matches (deaths successfully mapped to a telemetry point)
        matched_deaths = mapped.dropna(subset=['telem_unique_id'])
        print(f"Deaths mapped to telemetry windows: {len(matched_deaths)} / {len(df_d)}")
        
        # Count deaths per Telemetry ID
        counts = matched_deaths.groupby('telem_unique_id').size().reset_index(name='death_count')
        
        # Merge counts back to original Telemetry
        final_df = pd.merge(df_t, counts, on='telem_unique_id', how='left')
        final_df['death_count'] = final_df['death_count'].fillna(0).astype(int)
        
        # Clean temp
        cols_drop = ['num_time', 'dt_time', 'telem_unique_id', 'dt_time_d', 'dt_time_t']
        final_df.drop(columns=[c for c in cols_drop if c in final_df.columns], inplace=True, errors='ignore')
        
        # Export
        os.makedirs('data/processed', exist_ok=True)
        final_df.to_csv(OUT_Processed, index=False)
        print(f"Exported processed data to {OUT_Processed} with {len(final_df)} rows.")
        
        # Analysis
        multi_deaths = final_df[final_df['death_count'] > 1]
        if len(multi_deaths) > 0:
            print(f"\nTelemetry Rows with > 1 death in a 30s window: {len(multi_deaths)}")
            print("Telemetry IDs (first 50):")
            if '_id' in multi_deaths.columns:
                print(multi_deaths['_id'].head(50).tolist())
            else:
                print(multi_deaths.index.head(50).tolist())
        else:
            print("\nNo telemetry windows found with > 1 death.")
else:
    print("Error: Data files not found.")

Loading datasets for death analysis...
Time Unit: s, Tolerance: 30
Deaths mapped to telemetry windows: 240 / 241
Exported processed data to data/processed/telemetry_with_deaths.csv with 3348 rows.

Telemetry Rows with > 1 death in a 30s window: 65
Telemetry IDs (first 50):
['69748e0d48d53c4152cf20ef', '697491f748d53c4152cf2c26', '697495d648d53c4152cf3771', '6974f7e55d8e98b89f0a69f4', '6974fb615d8e98b89f0a8065', '6974ff545d8e98b89f0a9373', '697509445d8e98b89f0aa60a', '69750afa5d8e98b89f0aab41', '69750b105d8e98b89f0aaba1', '69750ee45d8e98b89f0acc68', '69750ef25d8e98b89f0accd3', '697517565d8e98b89f0afa39', '69758bf4547a2b541d10f7b9', '697593b9547a2b541d110c2e', '697593cc547a2b541d110c9b', '697597ae547a2b541d112ec0', '69759b8f547a2b541d11577a', '69759b91547a2b541d1157d3', '69759b95547a2b541d115811', '69759d07547a2b541d116de7', '69759f68547a2b541d11945c', '69759f6a547a2b541d1194ce', '69759fa9547a2b541d119812', '6975a365547a2b541d11d34e', '6975a377547a2b541d11d497', '6975a96d547a2b541d11ed30